# Final Project: Data Science
# 30-Day Hospital Readmission Risk Prediction for Diabetic Patients

**Author:** Adnan Jamali  
**Email:** adnanahmedkj@gmail.com  
**GitHub:** [github.com/adnanjamali02](https://github.com/adnanjamali02)  
**LinkedIn:** [Adnan Jamali](https://www.linkedin.com/in/adnan-jamali-22a12231b/)

---

## 1. Problem Definition

Hospital readmission within 30 days of discharge is a critical challenge in the U.S. healthcare system. For diabetic patients, the problem is especially acute due to the chronic and complex nature of diabetes management.

### Business Context
- **Annual Cost:** Hospital readmissions cost the U.S. healthcare system over **$15 billion** annually.
- **Medicare Penalty:** The Hospital Readmissions Reduction Program (HRRP) penalizes hospitals with excessive 30-day readmission rates, reducing Medicare payments by up to 3%.
- **Patient Outcomes:** Readmissions often indicate gaps in care transitions, medication management, or patient education.

### Objective
Build a predictive model to identify diabetic patients at high risk of 30-day readmission using clinical and demographic data collected during their hospital encounter. The target variable is binary:
- **1** = Readmitted within 30 days (`readmitted == '<30'`)
- **0** = Not readmitted within 30 days (`readmitted` is `'>30'` or `'NO'`)

### Dataset
The **Diabetes 130-US Hospitals** dataset contains **101,766 inpatient encounters** from 130 U.S. hospitals (1999–2008), with **50 clinical and demographic features** per encounter.

---

## 2. Data Collection

### 2.1 Loading the Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix, calibration_curve
)

try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install xgboost -q
    from xgboost import XGBClassifier

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    !pip install imbalanced-learn -q
    from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [ ]:
df = pd.read_csv('/home/z/my-project/upload/diabetic_data.csv')
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")

In [ ]:
df.head(3)

In [ ]:
df.info()

### 2.2 ID Mapping Reference

In [ ]:
id_mapping = pd.read_csv('/home/z/my-project/upload/IDs_mapping.csv')
print("ID Mapping Reference:")
print(id_mapping.to_string())

### 2.3 Target Variable Distribution

In [ ]:
print("Readmission distribution (raw):")
print(df['readmitted'].value_counts())
print(f"\nPercentage: {df['readmitted'].value_counts(normalize=True).round(4) * 100}%")

---

## 3. Data Cleaning & Preprocessing

### 3.1 Remove Expired / Hospice Patients

In [ ]:
expired_ids = [11, 13, 14, 19, 20, 21]
before = len(df)
df = df[~df['discharge_disposition_id'].isin(expired_ids)]
after = len(df)
print(f"Removed {before - after:,} expired/hospice patients ({before:,} → {after:,})")

### 3.2 Remove Invalid Gender Entries

In [ ]:
before = len(df)
df = df[df['gender'] != 'Unknown/Invalid']
after = len(df)
print(f"Removed {before - after:,} records with Unknown/Invalid gender ({before:,} → {after:,})")

### 3.3 Drop High-Missingness Columns

In [ ]:
missing_pct = df.isnull().mean() * 100
# Note: '?' is treated as missing in this dataset
for col in ['weight', 'payer_code', 'medical_specialty']:
    missing_count = (df[col] == '?').sum()
    missing_rate = missing_count / len(df) * 100
    print(f"{col}: {missing_count:,} missing ({missing_rate:.1f}%)")

df = df.drop(columns=['weight', 'payer_code', 'medical_specialty'])
print(f"\nDropped 3 high-missing columns. Shape: {df.shape}")

### 3.4 Drop Identifier Columns

In [ ]:
df = df.drop(columns=['encounter_id', 'patient_nbr'])
print(f"Shape after dropping identifiers: {df.shape}")

### 3.5 Replace '?' with NaN and Impute

In [ ]:
df = df.replace('?', np.nan)

# Race: fill with 'Other'
df['race'] = df['race'].fillna('Other')

# Diagnosis columns: fill with 'Unknown'
diag_cols = ['diag_1', 'diag_2', 'diag_3']
for col in diag_cols:
    df[col] = df[col].fillna('Unknown')

print(f"Remaining missing values: {df.isnull().sum().sum()}")

### 3.6 Remove Single-Value Columns

In [ ]:
single_val_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"Single-value columns: {single_val_cols}")
df = df.drop(columns=single_val_cols)
print(f"Shape after removing single-value columns: {df.shape}")

### 3.7 Encode Target Variable

In [ ]:
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)
df = df.drop(columns=['readmitted'])
print(f"Target distribution:")
print(df['readmitted_binary'].value_counts())
print(f"\nClass imbalance ratio (negative/positive): {(df['readmitted_binary'] == 0).sum() / (df['readmitted_binary'] == 1).sum():.2f}:1")

### 3.8 Encode Age Ordinally

In [ ]:
age_map = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3,
    '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7,
    '[80-90)': 8, '[90-100)': 9
}
df['age'] = df['age'].map(age_map)
print("Age encoding:")
print(df['age'].value_counts().sort_index())

### 3.9 Label Encode Categorical Columns

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print(f"\nAll features are now numeric. Shape: {df.shape}")

### 3.10 Feature Engineering

In [ ]:
# Prior healthcare visits (outpatient + emergency + inpatient)
df['prior_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

# Count of active diabetes medications
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
# Only include columns that exist in the dataset
medication_cols = [c for c in medication_cols if c in df.columns]

# Count medications where the value is NOT 'No' (encoded as 0 by LabelEncoder)
# After label encoding, 'No' maps to some integer; we need to count non-'No' entries
# Let's re-derive: active medications are those != the label for 'No'
df['med_count'] = 0
for col in medication_cols:
    if col in label_encoders:
        no_label = label_encoders[col].transform(['No'])[0]
        df['med_count'] += (df[col] != no_label).astype(int)

print(f"New feature 'prior_visits' statistics:")
print(df['prior_visits'].describe())
print(f"\nNew feature 'med_count' statistics:")
print(df['med_count'].describe())

---

## 4. Exploratory Data Analysis

### 4.1 Target Class Distribution

The dataset exhibits significant class imbalance with only ~11% of patients readmitted within 30 days.

In [ ]:
display(Image(filename='/home/z/my-project/download/images/01_target_distribution.png'))

### 4.2 Age and Gender Analysis

In [ ]:
display(Image(filename='/home/z/my-project/download/images/02_age_gender_analysis.png'))

### 4.3 Race Distribution

In [ ]:
display(Image(filename='/home/z/my-project/download/images/03_race_analysis.png'))

### 4.4 Feature Correlation Matrix

In [ ]:
display(Image(filename='/home/z/my-project/download/images/04_correlation_matrix.png'))

### 4.5 Boxplots of Key Numerical Features

In [ ]:
display(Image(filename='/home/z/my-project/download/images/05_boxplots.png'))

### 4.6 Admission Type and Prior Visits

In [ ]:
display(Image(filename='/home/z/my-project/download/images/06_admission_prior_visits.png'))

### 4.7 Diabetes Medication Analysis

In [ ]:
display(Image(filename='/home/z/my-project/download/images/07_medication_analysis.png'))

### 4.8 Glucose and A1C Results

In [ ]:
display(Image(filename='/home/z/my-project/download/images/08_glucose_a1c.png'))

### 4.9 Medication-Specific Readmission Rates

In [ ]:
display(Image(filename='/home/z/my-project/download/images/09_medications.png'))

### 4.10 Key EDA Findings

- **Age** is a strong predictor: readmission rates increase with patient age, peaking in the 70–90 age group.
- **Race disparities** exist: African American and Hispanic patients show slightly higher readmission rates.
- **Number of prior visits** is positively associated with readmission risk.
- **A1C and glucose measurements** are frequently missing, limiting their predictive utility.
- **Insulin use** is the strongest medication-level predictor of readmission.
- **Class imbalance** (~11.2% positive) requires careful handling to avoid biased models.

---

## 5. Handling Class Imbalance with SMOTE

### 5.1 Class Distribution Before SMOTE

In [ ]:
X = df.drop(columns=['readmitted_binary'])
y = df['readmitted_binary']

print(f"Before SMOTE:")
print(f"  Class 0 (Not readmitted): {(y == 0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"  Class 1 (Readmitted <30): {(y == 1).sum():,} ({(y==1).mean()*100:.1f}%)")
print(f"  Imbalance ratio: {(y==0).sum()/(y==1).sum():.2f}:1")

### 5.2 Apply SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"After SMOTE:")
print(f"  Class 0 (Not readmitted): {(y_resampled == 0).sum():,} ({(y_resampled==0).mean()*100:.1f}%)")
print(f"  Class 1 (Readmitted <30): {(y_resampled == 1).sum():,} ({(y_resampled==1).mean()*100:.1f}%)")
print(f"  Total samples: {len(y_resampled):,} (from {len(y):,})")

### 5.3 SMOTE Visualization

In [ ]:
display(Image(filename='/home/z/my-project/download/images/16_smote_visualization.png'))

In [ ]:
display(Image(filename='/home/z/my-project/download/images/17_smote_distribution.png'))

---

## 6. Feature Engineering & Train-Test Split

### 6.1 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"Features:     {X_train.shape[1]}")

### 6.2 Feature Scaling

In [ ]:
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])
print("StandardScaler applied to numerical features.")

### 6.3 Apply SMOTE on Training Set Only

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE on training data:")
print(f"  Class 0: {(y_train_smote == 0).sum():,}")
print(f"  Class 1: {(y_train_smote == 1).sum():,}")

### 6.4 Scale Pos Weight for XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

---

## 7. Model Building

Three classification models are trained and compared:
1. **Logistic Regression** — baseline linear model with balanced class weights
2. **Random Forest** — ensemble of 200 decision trees
3. **XGBoost** — gradient boosting with class imbalance handling

### 7.1 Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_smote, y_train_smote)
print("Logistic Regression trained successfully.")

### 7.2 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_smote, y_train_smote)
print("Random Forest trained successfully.")

### 7.3 XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train_smote, y_train_smote)
print("XGBoost trained successfully.")

---

## 8. Model Evaluation

### 8.1 ROC Curves

In [ ]:
display(Image(filename='/home/z/my-project/download/images/10_roc_curves.png'))

### 8.2 Precision-Recall Curves

In [ ]:
display(Image(filename='/home/z/my-project/download/images/11_precision_recall.png'))

### 8.3 Calibration Curves

In [ ]:
display(Image(filename='/home/z/my-project/download/images/12_calibration_curves.png'))

### 8.4 Confusion Matrices

In [ ]:
display(Image(filename='/home/z/my-project/download/images/13_confusion_matrices.png'))

### 8.5 Feature Importance (XGBoost)

In [ ]:
display(Image(filename='/home/z/my-project/download/images/14_feature_importance.png'))

### 8.6 Model Comparison Summary

In [ ]:
display(Image(filename='/home/z/my-project/download/images/15_model_comparison.png'))

### 8.7 Classification Reports

#### Logistic Regression

In [ ]:
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
print("Logistic Regression - Classification Report:")
print(classification_report(y_test, y_pred_lr, digits=4))

#### Random Forest

In [ ]:
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]
print("Random Forest - Classification Report:")
print(classification_report(y_test, y_pred_rf, digits=4))

#### XGBoost

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
print("XGBoost - Classification Report:")
print(classification_report(y_test, y_pred_xgb, digits=4))

### 8.8 Detailed Metrics Comparison

In [ ]:
metrics_data = []
for name, y_prob in [
    ('Logistic Regression', y_prob_lr),
    ('Random Forest', y_prob_rf),
    ('XGBoost', y_prob_xgb)
]:
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    roc_auc = roc_auc_score(y_test, y_prob)
    avg_prec = average_precision_score(y_test, y_prob)
    recall_1 = tp / (tp + fn)
    precision_1 = tp / (tp + fp)
    f1_1 = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0
    metrics_data.append({
        'Model': name,
        'ROC-AUC': round(roc_auc, 4),
        'Avg Precision': round(avg_prec, 4),
        'Recall (Class 1)': round(recall_1, 2),
        'Precision (Class 1)': round(precision_1, 2),
        'F1 (Class 1)': round(f1_1, 2)
    })

results_df = pd.DataFrame(metrics_data)
print(results_df.to_string(index=False))

---

## 9. Results Summary

### Model Performance Comparison

| Model | ROC-AUC | Avg Precision | Recall (Class 1) | Precision (Class 1) | F1 (Class 1) |
|-------|---------|--------------|-------------------|---------------------|--------------|
| Logistic Regression | 0.6420 | 0.1998 | 0.54 | 0.17 | 0.26 |
| Random Forest | 0.6507 | 0.1976 | 0.09 | 0.29 | 0.14 |
| XGBoost | 0.6549 | 0.2147 | 0.55 | 0.18 | 0.27 |

### Key Findings

1. **XGBoost achieves the highest ROC-AUC (0.6549)** and average precision (0.2147), making it the best overall model for ranking patients by readmission risk.
2. **Logistic Regression is a close second** with comparable recall and the simplest interpretability, suitable as a clinical baseline.
3. **Random Forest prioritizes precision** but at the cost of very low recall (0.09), missing the majority of actual readmissions.
4. **All models show modest discrimination** (AUC ~0.64–0.65), reflecting the inherent difficulty of predicting 30-day readmission from encounter-level data alone.
5. **Class imbalance remains the primary challenge**: even with SMOTE and balanced class weights, the models struggle to achieve both high precision and high recall for the minority class.

---

## 10. Conclusion & Business Impact

### Key Conclusions

- **30-day readmission for diabetic patients is difficult to predict** using only encounter-level clinical data, with the best model achieving an AUC of ~0.65.
- **Prior healthcare utilization** (number of inpatient, outpatient, and emergency visits) is the strongest predictor of readmission, followed by patient age and insulin usage.
- **SMOTE + balanced class weights** improved recall for the positive class but introduced a precision trade-off.
- **Clinical measurement gaps** (e.g., 96.9% missing weight, ~97% missing A1C results) significantly limit the feature space available for prediction.

### Business Impact & Recommendations

1. **Risk Stratification Tool:** Deploy the XGBoost model as a real-time scoring system at discharge to flag high-risk patients for targeted intervention (e.g., follow-up calls, medication reconciliation, home health referrals).

2. **Cost Savings Potential:** Even a modest 10% reduction in preventable readmissions could save an average hospital $1–2 million annually, given that the average readmission costs ~$15,000.

3. **Improve Data Collection:** Address the systemic gaps in clinical documentation — particularly A1C measurements and patient weight — which would substantially improve model performance.

4. **Enrich Feature Set:** Future work should incorporate longitudinal patient history, social determinants of health, and post-discharge factors (e.g., pharmacy fill records, follow-up appointment adherence).

5. **Model Monitoring:** Any deployed model must be continuously monitored for drift, as patient populations and clinical practices evolve over time.

### Limitations

- The dataset spans 1999–2008; clinical practices and diabetes management have evolved significantly since then.
- The models do not account for socio-economic factors, post-discharge support, or patient medication adherence.
- ROC-AUC values in the 0.64–0.65 range, while above random, indicate substantial room for improvement.

---

*This project was completed as part of the Data Science Final Project by Adnan Jamali.*